# creds.py

```python
GITHUB_ACCESS_TOKEN="my-github-classic-token"
GITHUB_USER_NAME="github-username"
GITHUB_REPO="github-repo"
```

# conftest.py

```python
import pytest
from creds import *
from playwright.sync_api import Playwright, APIRequestContext


@pytest.fixture(scope="session")
def api_context(playwright: Playwright):
    context = playwright.request.new_context(
        base_url="github-url.com",
        extra_http_headers={
            "Accept": "application/vnd.github.v3+json",
            "Authorization": f"token {GITHUB_ACCESS_TOKEN}"
        }
    )

    yield context
    context.dispose()

@pytest.fixture(autouse=True, scope="session")
def create_test_repository(api_context: APIRequestContext):

    # Create the test repo
    post_response = api_context.post(
        "/user/repos",
        data={"name": GITHUB_REPO}
    )
    assert post_response.ok

    yield

    # Delete the test repo
    delete_response = api_context.delete(
        f"/repos/{GITHUB_USER_NAME}/{GITHUB_REPO}"
    )
    assert delete_response.ok
```

# test_github_issue.py

```python

from playwright.sync_api import APIRequestContext, Page

from creds import *


def test_create_issue(api_context: APIRequestContext):
    issue_data = {
        "title": "[BUG] That Went Wrong",
        "body": "When doing this, that failed"
    }

    post_response = api_context.post(
        f"/repos/{GITHUB_USER_NAME}/{GITHUB_REPO}/issues",
        data=issue_data
    )

    assert post_response.ok

def test_issue_page(page: Page):
    page.goto(f"https://github.com/{GITHUB_USER_NAME}/{GITHUB_REPO}/issues")
    page.screenshot(path="issues-page.jpg", full_page=True)

def test_new_issue_in_repo(api_context: APIRequestContext):
    all_issues = api_context.get(
        f"repos/{GITHUB_USER_NAME}/{GITHUB_REPO}/issues"
    )

    assert all_issues.ok

    new_issue = [
        issue
        for issue in all_issues.json
        if issue["title"] == "[BUG] That Went Wrong"
    ][0]

    assert new_issue["body"] == "When doing this, that failed"

```